In [1]:
# ── Core ────────────────────────────────────────────────
import os
import re
import json
import random
import warnings
from pathlib import Path

# ── Data ────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualisation ───────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns

# ── Machine Learning utilities ──────────────────────────
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# ── Deep Learning ───────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ── Transformers ────────────────────────────────────────
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)

# ── Configuration ───────────────────────────────────────
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PALETTE = ["#FF6B9D", "#4ABCE8", "#F5A623"]
sns.set_theme(style="whitegrid", palette=PALETTE)

LANGUAGES = ["en", "es", "pt", "it"]


# The Grammar of Luxury

### A multilingual model of luxury sentiment across guest reviews

---

> *"Some people think luxury is the opposite of poverty. It is not. It is the opposite of vulgarity."*
>
> — attributed to Coco Chanel

---

**Author:** Avgustina Daskalova
**Course:** SoftUni — Deep Learning, July 2026
**Repository:** [github.com/Gusti4ka/The-Grammar-of-Luxury](https://github.com/Gusti4ka/The-Grammar-of-Luxury)

---

## Abstract

Chanel drew the line not between the rich and the poor, but between the discreet and the vulgar — and in doing so, she made luxury a matter of *register*. Of what is said, and what is left unsaid. Of tone rather than price.

Register is a linguistic property. And linguistic properties do not survive translation intact.

A Brazilian guest writes *aconchegante*; an Italian, *raffinato*; an Englishwoman, *understated*. Each word arrives carrying its own grammar of expectation — a private arithmetic of what refinement means and how one is permitted to speak of it. The words are not synonyms. They are dialects of the same discretion.

This project asks whether a machine can learn that grammar across borders.

It stands on two prior works. **Lost in Translation** established that semantic translation loss $\mathcal{L}(c)$ is *instrument-dependent*: monolingual embeddings collapsed under the weight of crossing languages, while multilingual ones held. **Arte Discreta** built a 74-term taxonomy of luxury across five aspects — but spoke only English, and asked a binary question of a dataset holding two positive answers. An honest null result; a hypothesis the data could not test.

*The Grammar of Luxury* answers both.

The taxonomy becomes the instrument rather than the subject: through distant supervision, its 74 terms generate a **continuous aspect-sentiment score**, dissolving the small-$n$ constraint that silenced its predecessor. Contextual multilingual embeddings — LaBSE and XLM-R, the instruments *Lost in Translation* vindicated — are then fine-tuned to read that score across **English, Spanish, Portuguese, and Italian**.

A second trained model closes the circuit. A fine-tuned voice, narrating each report in the language it was written in — text returning to speech, the unstructured become audible.

Two modalities. Four languages. One grammar.

---

## Table of Contents

1. **[Data — The Multilingual Corpus](#1-data--the-multilingual-corpus)**
   - 1.1 MAiDE-up (English, Spanish, Italian)
   - 1.2 Brazilian Hotel Reviews (Portuguese)
   - 1.3 Unified Corpus
2. **[The Taxonomy of Luxury](#2-the-taxonomy-of-luxury)** — porting Arte Discreta's 74 terms
3. **[Distant Supervision](#3-distant-supervision)** — from taxonomy to continuous luxury score
4. **[Exploratory Analysis](#4-exploratory-analysis)** — the corpus at a glance
5. **[The Text Model](#5-the-text-model)** — fine-tuning LaBSE / XLM-R
   - 5.1 Tokenisation & Datasets
   - 5.2 Architecture
   - 5.3 Training
   - 5.4 Evaluation
6. **[Interpretability](#6-interpretability)** — attention across languages
7. **[The Voice Model](#7-the-voice-model)** — narrating the report
8. **[Conclusions](#8-conclusions)**
9. **[References](#9-references)**

---

## 1. Data — The Multilingual Corpus

Two datasets feed this project, one per source-language group.

### 1.1 MAiDE-up (English, Spanish, Italian)

The **MAiDE-up** corpus (Ignat et al., 2024) collects 20,000 hotel reviews from Booking.com across ten languages, half genuine and half generated by GPT-4 for a deception-detection task. Three of its languages — **English, Spanish, Italian** — are ours.

Two decisions shape how we use it:

- **Real reviews only.** The corpus was built to distinguish human from machine writing. That is not our question. A model of *luxury* must learn from how real guests actually speak, so we discard the AI-generated half and keep only the human-written reviews (`source == 0`).

- **Reunited text.** Each review is stored split across two columns — `Upside_Review` and `Downside_Review`. We recombine them into a single passage, since luxury sentiment lives in the whole utterance, not in praise or complaint alone.

In [2]:
# ── Download MAiDE-up dataset (10 languages; real + AI reviews) ──
import urllib.request

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

MAIDE_URL = (
    "https://raw.githubusercontent.com/MichiganNLP/"
    "multilingual_reviews_deception/master/data/all_data.csv"
)
maide_path = DATA_DIR / "maide_up_all.csv"

if not maide_path.exists():
    print("Downloading MAiDE-up dataset…")
    urllib.request.urlretrieve(MAIDE_URL, maide_path)
    print(f"Saved to {maide_path}")
else:
    print(f"Already present: {maide_path}")
    

Already present: data\maide_up_all.csv


In [3]:
# ── Load MAiDE-up and inspect structure ──
maide = pd.read_csv(maide_path)

print(f"Shape: {maide.shape[0]:,} rows × {maide.shape[1]} columns\n")
print("Columns:", list(maide.columns), "\n")
maide.head(3)

Shape: 19,985 rows × 12 columns

Columns: ['Unnamed: 0', 'Review_Language', 'City Name', 'Hotel Name', 'Upside_Review', 'Downside_Review', 'Review_Score', 'Sentiment', 'source', 'Prompt_Language', 'na_up_review', 'na_down_review'] 



,Unnamed: 0,Review_Language,City Name,Hotel Name,Upside_Review,Downside_Review,Review_Score,Sentiment,source,Prompt_Language,na_up_review,na_down_review
0,0,Chinese,Ankara,迪万卡拉汉酒店,性价比超高的酒店，虽然价格比较贵，但是享受到的服务环境都还不错。,NaN,10.0,POS,0,NaN,False,True
1,1,Chinese,Ankara,迪万卡拉汉酒店,奥斯曼风格的建筑很有特色，房间装修精致，距离安卡拉城堡非常近，附近餐厅多，纪念品小商店多,不知道是不是因为夏季的原因，房间的空调不太给力，房间热；酒店员工会在窗外聊天，不考虑客人是不...,4.0,NEG,0,NaN,False,False
2,2,Chinese,Ankara,迪万卡拉汉酒店,非常喜歡酒店的環境氛圍，特別是通過地下通道到達博物館，這是一個驚喜（但入住時酒店服務人員并未...,酒店的空調令人傷心，第一天房間空調不起作用，次日上午出門時告知酒店經理，回來后依然沒有處理，...,5.0,NEG,0,NaN,False,False


In [4]:
# ── Filter: real reviews only, in EN / ES / IT ──
LANG_NAMES = {"English": "en", "Spanish": "es", "Italian": "it"}

# Keep real reviews (source == 0) in our three MAiDE-up languages
mask = (maide["source"] == 0) & (maide["Review_Language"].isin(LANG_NAMES))
maide_real = maide[mask].copy()

# Combine upside + downside into one review text
maide_real["text"] = (
    maide_real["Upside_Review"].fillna("").astype(str) + " "
    + maide_real["Downside_Review"].fillna("").astype(str)
).str.strip()

# Normalise language to short codes
maide_real["lang"] = maide_real["Review_Language"].map(LANG_NAMES)

# Keep only what we need
maide_clean = maide_real[["lang", "text", "Review_Score", "Sentiment"]].reset_index(drop=True)

print(f"Real EN/ES/IT reviews: {len(maide_clean):,}\n")
print(maide_clean["lang"].value_counts(), "\n")
maide_clean.head(3)

Real EN/ES/IT reviews: 3,000

lang
en    1000
it    1000
es    1000
Name: count, dtype: int64 



,lang,text,Review_Score,Sentiment
0,en,Staff. Serkan türk was a great help during our...,10.0,POS
1,en,Nicely furnished room and nicely decorated lob...,9.0,POS
2,en,"Very central, near 2 metro stations. Room was ...",9.0,POS


In [5]:
# ── Unzip the Brazilian Portuguese corpus ──
import zipfile

pt_zip = DATA_DIR / "tripadvisorBrazilianHotelREviews.zip"

with zipfile.ZipFile(pt_zip, "r") as z:
    print("Files inside the archive:")
    for name in z.namelist():
        print("  ", name)
    z.extractall(DATA_DIR)

print("\nExtracted.")

Files inside the archive:
   datesBrazil.txt
   gradesBrazil.txt
   titlesBrazil.txt
   commentsBrazilNormalized.txt

Extracted.


In [6]:
# ── Load the Brazilian Portuguese corpus (4 parallel text files) ──
def read_lines(filename):
    """Read a text file into a list of stripped lines."""
    with open(DATA_DIR / filename, encoding="utf-8") as f:
        return [line.strip() for line in f]

grades   = read_lines("gradesBrazil.txt")
titles   = read_lines("titlesBrazil.txt")
comments = read_lines("commentsBrazilNormalized.txt")
dates    = read_lines("datesBrazil.txt")

print(f"grades:   {len(grades):,}")
print(f"titles:   {len(titles):,}")
print(f"comments: {len(comments):,}")
print(f"dates:    {len(dates):,}")


grades:   730,069
titles:   730,069
comments: 730,069
dates:    730,069


In [7]:
# ── Build Portuguese DataFrame and subsample to match other languages ──
pt_full = pd.DataFrame({
    "grade": grades,
    "title": titles,
    "comment": comments,
    "date": dates,
})

print(f"Full Portuguese corpus: {len(pt_full):,} reviews")

# Combine title + comment into one review text (mirrors the MAiDE-up treatment)
pt_full["text"] = (pt_full["title"] + " " + pt_full["comment"]).str.strip()

# Drop empty or very short reviews
pt_full = pt_full[pt_full["text"].str.len() > 20]

# Subsample 1,000 to match EN / ES / IT
pt_clean = pt_full.sample(n=1000, random_state=SEED).reset_index(drop=True)
pt_clean["lang"] = "pt"

print(f"After cleaning:         {len(pt_full):,}")
print(f"Subsampled to:          {len(pt_clean):,}\n")
pt_clean[["lang", "text", "grade"]].head(3)


Full Portuguese corpus: 730,069 reviews
After cleaning:         730,066
Subsampled to:          1,000



,lang,text,grade
0,pt,"Quartos confortáveis mas had Para o preço, não...",2
1,pt,Sabe aquele velhinho que deram um tapa?? O lug...,3
2,pt,Aconchegante Situado muito próximo à praia dos...,4


### 1.2 Brazilian Hotel Reviews (Portuguese)

Portuguese is absent from MAiDE-up, so it arrives from a second source: the **Brazilian Portuguese Hotel's Reviews Corpus** (Souza, Oliveira & Moreira, 2018), 730,069 TripAdvisor reviews gathered from hotels across all 26 Brazilian state capitals and the Federal District.

The corpus is distributed as four parallel text files — grades, titles, comments, dates — one line per review. We recombine title and comment into a single passage, matching the treatment given to MAiDE-up's split reviews.

Two adjustments make it comparable to the others:

- **Subsampling.** 730,069 reviews would drown out the 1,000 available per language elsewhere. We draw a random sample of **1,000**, so no language dominates the model's attention.

- **Scale.** TripAdvisor grades reviews from 1 to 5; Booking.com, the source behind MAiDE-up, uses 1 to 10. These are normalised to a common scale in §1.3.

### 1.3 The Unified Corpus

Four languages, two sources, one corpus.

The two datasets arrive with different conventions, so unification requires two normalisations:

- **A common rating scale.** MAiDE-up carries Booking.com scores (1–10); the Brazilian corpus carries TripAdvisor circles (1–5). Both are mapped onto a shared 0–1 interval, preserving relative position while discarding the arbitrary units of each platform.

- **A common shape.** Each review is reduced to three fields: the language it was written in, its text, and its normalised rating. Everything else — hotel names, cities, dates, sentiment labels — falls away. What remains is what the model will read.

The result is a balanced quartet: 1,000 reviews in each of English, Spanish, Portuguese, and Italian. No language speaks louder than another.

In [8]:
# ── Unify: one corpus, four languages, common scale ──

# Normalise ratings to 0–1
maide_part = maide_clean[["lang", "text"]].copy()
maide_part["rating"] = maide_clean["Review_Score"] / 10.0        # Booking.com 1–10

pt_part = pt_clean[["lang", "text"]].copy()
pt_part["rating"] = pt_clean["grade"].astype(float) / 5.0        # TripAdvisor 1–5

# Combine
corpus = pd.concat([maide_part, pt_part], ignore_index=True)
corpus = corpus.sample(frac=1, random_state=SEED).reset_index(drop=True)  # shuffle

print(f"Unified corpus: {len(corpus):,} reviews\n")
print(corpus["lang"].value_counts(), "\n")
print(corpus["rating"].describe(), "\n")
corpus.head(5)

Unified corpus: 4,000 reviews

lang
en    1000
pt    1000
es    1000
it    1000
Name: count, dtype: int64 

count    4000.000000
mean        0.704000
std         0.261297
min         0.100000
25%         0.400000
50%         0.800000
75%         0.900000
max         1.000000
Name: rating, dtype: float64 



,lang,text,rating
0,en,Room was spacious. Had sofas and a table. Clea...,0.4
1,pt,"AGRADÁVEL SURPRESA!! Esse hotel é bem novo, co...",1.0
2,en,It is very close to the VFS and Shivaji stadiu...,0.4
3,pt,Para viagem de negócios! Para o viajante à neg...,0.8
4,es,"Habitaciones muy cómodas y muy completas, exce...",0.9
